# Module 3 Hands-On Problem 2.1: Outliers in the Crime Data

**Name:** Mac Mckay
**Course:** CS 82A
**Repo:** https://github.com/raulmckay-blip/cs82a-portfolio

## Overview

This lab looks at a real crime dataset (`usarrests.csv`) to find values that can't actually be true. It has one row for each US state, with arrest rates per 100,000 people for murder and assault, and the percent of each state's population living in urban areas.

There are two kinds of bad values to check for:

- **Impossible**: the value breaks the definition of the measurement. A percent over 100 is a good example, since a percent can never go higher than 100.
- **Extreme**: the value is just unusual, not against any rule. It's a value that could really happen, just far from the rest.

If a value is impossible, it's an error, so I fix it. If a value is extreme, it's real data, so I keep it.

## Approach

I did this the same way I did Lab 1: count, look closer, and write it down. I used summary statistics to check each column, filtered out the rows that looked off, and decided whether each flagged value was impossible or extreme. I wrote down what I did about each one below.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("usarrests.csv")
df = df.rename(columns={"Unnamed: 0": "State"})
print(df.shape)
df.head()

## Summary Statistics

In [ ]:
df.describe()

## Impossible Values

In [ ]:
df[df["UrbanPop"] > 100]

In [ ]:
num_cols = df.select_dtypes("number").columns
df[(df[num_cols] < 0).any(axis=1)]

## Extreme Values (IQR Check)

In [ ]:
for col in num_cols:
    q1, q3 = df[col].quantile([0.25, 0.75])
    upper = q3 + 1.5 * (q3 - q1)
    lower = q1 - 1.5 * (q3 - q1)
    flagged = df[(df[col] > upper) | (df[col] < lower)]
    if len(flagged) > 0:
        print(f"{col}: outside {lower:.1f} to {upper:.1f}")
        display(flagged)

## Handling the Impossible Value

In [ ]:
df.loc[df["UrbanPop"] > 100, "UrbanPop"] = np.nan
df.describe()

## Flagged Values

Looking at `df.describe()` and the IQR check, I found four values worth flagging: one impossible, two extreme, and one missing value that was already in the data before I touched it.

**1. Iowa, UrbanPop = 570. Impossible.**
UrbanPop is a percentage, so it can't go above 100. A value of 570 is more than five times too high, which is also why the column's max (570) is so much bigger than its 75th percentile (77.75) in the summary table. This is most likely a data entry mistake, maybe an extra digit or a decimal point in the wrong spot. I replaced it with NaN instead of deleting the whole row so Iowa's other numbers are kept.

**2. South Carolina, Assault = 879. Extreme.**
The IQR check flags anything above 459 as outside the normal range for this column, and 879 goes way past that. It's almost double the next highest state, Florida, at 335. Even so, it's still a real, possible arrest rate. Nothing about the measurement rules out a number that high. I kept it because taking out real extreme values would hide actual differences between states.

**3. New York, UrbanPop = 6. Extreme.**
The same IQR check flags anything below 16.5 as unusual, and 6 is well under that. It's still a real percentage, somewhere between 0 and 100, so it doesn't count as impossible. But a 6 percent urban population for a state like New York doesn't match what I'd expect in real life. I kept the value, but I'm noting it here as one I'd want to double check against a real source before using it in any analysis.

**4. Georgia, Assault. Missing, not impossible or extreme.**
This one doesn't really fit the other two categories. It's not a bad value, it's just not there. The count for `Assault` in the summary table is 49 instead of 50, and checking that showed Georgia's `Assault` cell is blank in the raw file. I didn't cause this by cleaning the data. It was already missing before I changed anything, so I left it as NaN instead of guessing what it should be.